<a href="https://colab.research.google.com/github/kamathvk1982/GAI601/blob/main/AssignmentThree-ModelTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment 3: Experimentation & Model Training**

Author(s): Vinayak Kamath

Course/Subject: GAI 601 Machine Learning

Date of Submission: April 26th , 2026

**Introduction**

Data Science is called a science because it follows the scientific method. In Machine Learning, Experimentation refers to the systematic process of designing, running, and analyzing different configurations to identify the optimal settings that performs best. Experimentation is learning by doing. Experimentation involves changing parameters, evaluating results (using metrics), and learning from each experiment to iteratively find the optimal model.

**Do the following:**

1.	Experiment (train a model) using one of the Decision Tree algorithms (or feel free to use any other you like), and for every experiment do the following:

    a.	Planning - the Hypothesis - Define the objective of the experiment. What are you trying to learn? What is your theory & how will you test it?
    
    b.	Planning - Change one thing - Decide what you will change to test that hypothesis. Experiments should modify only one parameter or algorithm at a time. If the results improve you are on the right track, otherwise your hypothesis is wrong.
    
    c.	Train a model - the "Experiment" - Train the model i.e. run an experiment.
    
    d.	Evaluation - Was you hypothesis true or not? What did you learn?
    
    e.	Planning - Recommendation - What do you recommend for the next experiment (based on the evaluation?). What is your new hypothesis?

2.	Run 6 experiments in total - ensuring that you build on top of each experiment (by learning from the previous one)



## Experiment 1: Basic Decision Tree Classifier

### 1.1. Data Loading

First, we'll load the `bank-full.csv` dataset. This dataset is semicolon-separated.

In [7]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/kamathvk1982/GAI601/main/bank-full.csv"
df = pd.read_csv(csv_url, sep=';')

print("Data loaded successfully. Displaying first 5 rows:")
display(df.head())

Data loaded successfully. Displaying first 5 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


### 1.2. Data Preprocessing

For our first experiment, we will predict the `y` column (whether the client subscribed to a term deposit). We will preprocess the data by dropping the `duration` column (to prevent target leakage) and encoding categorical features.

In [8]:
from sklearn.preprocessing import LabelEncoder

df_processed = df.copy()

# Drop 'duration' to avoid target leakage (if predicting before the call outcome)
columns_to_drop = ['duration']
df_processed = df_processed.drop(columns=[col for col in columns_to_drop if col in df_processed.columns])

# Encode categorical features
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if 'y' in categorical_cols: # Ensure 'y' is not encoded as a feature yet
    categorical_cols.remove('y')

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])

# Define features (X) and target (y)
y = df_processed['y']
X = df_processed.drop(columns=['y'])

# Encode target variable 'y' ('yes'/'no' to 0/1)
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print("Processed data head:")
display(X.head())
print("Target variable (encoded) value counts:")
display(pd.Series(y_encoded).value_counts())

Processed data head:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome
0,58,4,1,2,0,2143,1,0,2,5,8,1,-1,0,3
1,44,9,2,1,0,29,1,0,2,5,8,1,-1,0,3
2,33,2,1,1,0,2,1,1,2,5,8,1,-1,0,3
3,47,1,1,3,0,1506,1,0,2,5,8,1,-1,0,3
4,33,11,2,3,0,1,0,0,2,5,8,1,-1,0,3


Target variable (encoded) value counts:


,count
0,39922
1,5289


### 1.3. Planning - The Hypothesis

**Hypothesis:** A Decision Tree Classifier with default parameters can predict whether a client will subscribe to a term deposit with reasonable accuracy, utilizing the available numerical and label-encoded categorical features after basic preprocessing.

**Objective:** To establish a baseline performance for predicting the target variable `y` using a simple Decision Tree model.

### 1.4. Planning - Change One Thing

For this first experiment, we will use the `DecisionTreeClassifier` from `sklearn.tree` with its default parameters. This serves as our starting point.

### 1.5. Train a Model - The "Experiment"

We will split the preprocessed data into training and testing sets, then train the Decision Tree Classifier.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# Initialize and train the Decision Tree Classifier with default parameters
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

print("Model training complete.")

Training set size: 31647 samples
Testing set size: 13564 samples
Model training complete.


### 1.6. Evaluation - Was your hypothesis true or not? What did you learn?

Now, let's evaluate the performance of our default Decision Tree Classifier on the test set.

In [10]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=le_target.classes_)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

if accuracy > 0.5: # Simple heuristic for 'reasonable accuracy'
    print("\nBased on the accuracy, the hypothesis seems plausible. The default Decision Tree can predict client subscription to some extent.")
else:
    print("\nBased on the accuracy, the hypothesis might not be strongly supported. The default Decision Tree's performance is relatively low.")

print("\nWhat we learned: The baseline accuracy provides a starting point. We can see which classes are predicted well and which are not from the classification report.")

Accuracy: 0.8315

Classification Report:
              precision    recall  f1-score   support

          no       0.91      0.90      0.90     11966
         yes       0.30      0.33      0.31      1598

    accuracy                           0.83     13564
   macro avg       0.61      0.61      0.61     13564
weighted avg       0.84      0.83      0.83     13564


Based on the accuracy, the hypothesis seems plausible. The default Decision Tree can predict client subscription to some extent.

What we learned: The baseline accuracy provides a starting point. We can see which classes are predicted well and which are not from the classification report.


### 1.7. Planning - Recommendation for the next experiment

**Recommendation:** The default Decision Tree has provided a baseline. For the next experiment, we should explore tuning its hyperparameters to improve performance. A common hyperparameter to tune for Decision Trees is `max_depth` to control the complexity of the tree and prevent overfitting.

**New Hypothesis:** By optimizing the `max_depth` hyperparameter, we can improve the accuracy and generalization ability of the Decision Tree Classifier compared to the default model.

## Experiment 2: Decision Tree with Tuned `max_depth`

### 2.1. Planning - The Hypothesis

**Hypothesis:** By optimizing the `max_depth` hyperparameter using `GridSearchCV`, we can achieve a better balance between bias and variance, leading to improved accuracy and generalization performance of the Decision Tree Classifier compared to the default model from Experiment 1.

**Objective:** To find the optimal `max_depth` for the Decision Tree Classifier and assess its impact on model performance.

### 2.2. Planning - Change One Thing

For this experiment, we will systematically explore different values for the `max_depth` hyperparameter using `GridSearchCV` with cross-validation. We will keep all other parameters of the `DecisionTreeClassifier` at their defaults.

### 2.3. Train a Model - The "Experiment"

In [11]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for max_depth
param_grid = {
    'max_depth': [3, 5, 7, 10, 15, 20, None] # None means no limit
}

# Initialize a Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(random_state=42)

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=dt_classifier, param_grid=param_grid,
                           cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

print("GridSearchCV complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Get the best model
best_dt_model = grid_search.best_estimator_

Fitting 5 folds for each of 7 candidates, totalling 35 fits
GridSearchCV complete.
Best parameters found: {'max_depth': 5}
Best cross-validation accuracy: 0.8918


### 2.4. Evaluation - Was your hypothesis true or not? What did you learn?

In [12]:
# Make predictions with the best model
y_pred_tuned = best_dt_model.predict(X_test)

# Evaluate the tuned model
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
report_tuned = classification_report(y_test, y_pred_tuned, target_names=le_target.classes_)

print(f"Accuracy with tuned max_depth: {accuracy_tuned:.4f}")
print("\nClassification Report (Tuned Model):")
print(report_tuned)

# Compare with Experiment 1's accuracy
print(f"\nAccuracy of default Decision Tree (Experiment 1): {accuracy:.4f}")
print(f"Accuracy difference: {accuracy_tuned - accuracy:.4f}")

if accuracy_tuned > accuracy:
    print("\nHypothesis confirmed: Tuning max_depth improved the model's accuracy.")
else:
    print("\nHypothesis not strongly supported: Tuning max_depth did not significantly improve or even decreased accuracy.")

print("\nWhat we learned: We found the optimal max_depth and observed its effect on overall accuracy and class-wise performance. This helps in understanding the right complexity for our Decision Tree.")

Accuracy with tuned max_depth: 0.8912

Classification Report (Tuned Model):
              precision    recall  f1-score   support

          no       0.90      0.98      0.94     11966
         yes       0.62      0.20      0.31      1598

    accuracy                           0.89     13564
   macro avg       0.76      0.59      0.62     13564
weighted avg       0.87      0.89      0.87     13564


Accuracy of default Decision Tree (Experiment 1): 0.8315
Accuracy difference: 0.0596

Hypothesis confirmed: Tuning max_depth improved the model's accuracy.

What we learned: We found the optimal max_depth and observed its effect on overall accuracy and class-wise performance. This helps in understanding the right complexity for our Decision Tree.


### 2.5. Planning - Recommendation for the next experiment

**Recommendation:** We've seen the impact of `max_depth`. For the next experiment, we should investigate other hyperparameters that influence the Decision Tree's structure and generalization, such as `min_samples_split` or `min_samples_leaf`, or try combining `max_depth` with another hyperparameter.

**New Hypothesis:** By optimizing `min_samples_leaf` (or `min_samples_split`) in addition to or instead of `max_depth`, we can further prevent overfitting and improve the Decision Tree's predictive performance and robustness on unseen data.

## Experiment 3: Decision Tree with Tuned `max_depth` and `min_samples_leaf`

### 3.1. Planning - The Hypothesis

**Hypothesis:** By combining the optimal `max_depth` found in Experiment 2 with a tuned `min_samples_leaf` hyperparameter, we can further prevent overfitting and improve the Decision Tree's generalization performance and robustness on unseen data, potentially leading to a higher overall accuracy and/or better balanced class performance.

**Objective:** To find the optimal combination of `max_depth` and `min_samples_leaf` for the Decision Tree Classifier and assess its impact on model performance.

### 3.2. Planning - Change One Thing

For this experiment, we will use `GridSearchCV` to tune both `max_depth` (around the optimal value found in Experiment 2) and `min_samples_leaf`. All other parameters of the `DecisionTreeClassifier` will remain at their defaults.

### 3.3. Train a Model - The "Experiment"

In [13]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for max_depth and min_samples_leaf
# We'll refine the max_depth range around the best value from Experiment 2 (which was 5),
# and add min_samples_leaf.
param_grid_exp3 = {
    'max_depth': [4, 5, 6, 7, 8],
    'min_samples_leaf': [1, 5, 10, 20, 30, 50]
}

# Initialize a Decision Tree Classifier
dt_classifier_exp3 = DecisionTreeClassifier(random_state=42)

# Initialize GridSearchCV
grid_search_exp3 = GridSearchCV(estimator=dt_classifier_exp3, param_grid=param_grid_exp3,
                                cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search_exp3.fit(X_train, y_train)

print("GridSearchCV for Experiment 3 complete.")
print(f"Best parameters found for Experiment 3: {grid_search_exp3.best_params_}")
print(f"Best cross-validation accuracy for Experiment 3: {grid_search_exp3.best_score_:.4f}")

# Get the best model for Experiment 3
best_dt_model_exp3 = grid_search_exp3.best_estimator_

Fitting 5 folds for each of 30 candidates, totalling 150 fits
GridSearchCV for Experiment 3 complete.
Best parameters found for Experiment 3: {'max_depth': 4, 'min_samples_leaf': 1}
Best cross-validation accuracy for Experiment 3: 0.8923


### 3.4. Evaluation - Was your hypothesis true or not? What did you learn?

In [14]:
# Make predictions with the best model from Experiment 3
y_pred_exp3 = best_dt_model_exp3.predict(X_test)

# Evaluate the tuned model from Experiment 3
accuracy_exp3 = accuracy_score(y_test, y_pred_exp3)
report_exp3 = classification_report(y_test, y_pred_exp3, target_names=le_target.classes_)

print(f"Accuracy with tuned max_depth and min_samples_leaf: {accuracy_exp3:.4f}")
print("\nClassification Report (Experiment 3 Tuned Model):")
print(report_exp3)

# Compare with Experiment 2's accuracy
print(f"\nAccuracy of Decision Tree with tuned max_depth (Experiment 2): {accuracy_tuned:.4f}")
print(f"Accuracy difference (Exp 3 vs Exp 2): {accuracy_exp3 - accuracy_tuned:.4f}")

if accuracy_exp3 > accuracy_tuned:
    print("\nHypothesis confirmed: Tuning min_samples_leaf in addition to max_depth further improved the model's accuracy.")
elif accuracy_exp3 == accuracy_tuned:
    print("\nHypothesis partially confirmed: Tuning min_samples_leaf did not significantly change accuracy, but might offer other benefits (e.g., simpler tree).")
else:
    print("\nHypothesis not strongly supported: Tuning min_samples_leaf did not improve or even decreased accuracy.")

print("\nWhat we learned: We investigated the combined effect of max_depth and min_samples_leaf. This helps us understand how to control tree complexity more granularly.")

Accuracy with tuned max_depth and min_samples_leaf: 0.8916

Classification Report (Experiment 3 Tuned Model):
              precision    recall  f1-score   support

          no       0.90      0.99      0.94     11966
         yes       0.63      0.19      0.29      1598

    accuracy                           0.89     13564
   macro avg       0.77      0.59      0.62     13564
weighted avg       0.87      0.89      0.86     13564


Accuracy of Decision Tree with tuned max_depth (Experiment 2): 0.8912
Accuracy difference (Exp 3 vs Exp 2): 0.0004

Hypothesis confirmed: Tuning min_samples_leaf in addition to max_depth further improved the model's accuracy.

What we learned: We investigated the combined effect of max_depth and min_samples_leaf. This helps us understand how to control tree complexity more granularly.


### 3.5. Planning - Recommendation for the next experiment

**Recommendation:** We have tuned `max_depth` and `min_samples_leaf`. While overall accuracy has improved, the recall for the 'yes' class (subscribers) is still relatively low. For the next experiment, we should address this class imbalance, as Decision Trees can be sensitive to it.

**New Hypothesis:** By using techniques to handle class imbalance, such as adjusting class weights or oversampling/undersampling, we can improve the recall and F1-score for the minority class ('yes') without significantly sacrificing overall accuracy, leading to a more balanced and effective model.

## Experiment 4: Decision Tree with Tuned `max_depth`, `min_samples_leaf`, and Class Weighting

### 4.1. Planning - The Hypothesis

**Hypothesis:** By incorporating `class_weight='balanced'` into the Decision Tree Classifier, in addition to the tuned `max_depth` and `min_samples_leaf` from Experiment 3, we can improve the model's ability to correctly identify the minority class ('yes'), leading to a higher recall and F1-score for this class, without significantly compromising overall accuracy.

**Objective:** To mitigate the effect of class imbalance and achieve a more balanced performance across both classes, particularly improving the prediction of term deposit subscriptions.

### 4.2. Planning - Change One Thing

For this experiment, we will introduce `class_weight='balanced'` to the `DecisionTreeClassifier`. We will use the best `max_depth` and `min_samples_leaf` found in Experiment 3 as our starting point, and we'll re-run `GridSearchCV` to ensure optimal parameters with this new setting.

### 4.3. Train a Model - The "Experiment"

In [15]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid, focusing on max_depth and min_samples_leaf, now adding class_weight
param_grid_exp4 = {
    'max_depth': [4, 5, 6, 7, 8], # Around the best value from Experiment 3
    'min_samples_leaf': [1, 5, 10, 20, 30, 50], # Around the best value from Experiment 3
    'class_weight': [None, 'balanced'] # Introduce class weighting
}

# Initialize a Decision Tree Classifier with random_state
dt_classifier_exp4 = DecisionTreeClassifier(random_state=42)

# Initialize GridSearchCV for Experiment 4
grid_search_exp4 = GridSearchCV(estimator=dt_classifier_exp4, param_grid=param_grid_exp4,
                                cv=5, scoring='f1', n_jobs=-1, verbose=1) # Using 'f1' or 'recall' for imbalanced data

# Fit GridSearchCV to the training data
grid_search_exp4.fit(X_train, y_train)

print("GridSearchCV for Experiment 4 complete.")
print(f"Best parameters found for Experiment 4: {grid_search_exp4.best_params_}")
print(f"Best cross-validation F1-score for Experiment 4: {grid_search_exp4.best_score_:.4f}")

# Get the best model for Experiment 4
best_dt_model_exp4 = grid_search_exp4.best_estimator_

Fitting 5 folds for each of 60 candidates, totalling 300 fits
GridSearchCV for Experiment 4 complete.
Best parameters found for Experiment 4: {'class_weight': 'balanced', 'max_depth': 6, 'min_samples_leaf': 20}
Best cross-validation F1-score for Experiment 4: 0.4049


### 4.4. Evaluation - Was your hypothesis true or not? What did you learn?

In [16]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions with the best model from Experiment 4
y_pred_exp4 = best_dt_model_exp4.predict(X_test)

# Evaluate the tuned model from Experiment 4
accuracy_exp4 = accuracy_score(y_test, y_pred_exp4)
report_exp4 = classification_report(y_test, y_pred_exp4, target_names=le_target.classes_)

print(f"Accuracy with tuned max_depth, min_samples_leaf, and class weighting: {accuracy_exp4:.4f}")
print("\nClassification Report (Experiment 4 Tuned Model):")
print(report_exp4)

# Compare with Experiment 3's metrics (especially recall for 'yes')
# For simplicity, we can use the reported metrics from previous cell's output or re-evaluate best_dt_model_exp3
# Let's re-evaluate best_dt_model_exp3 for a direct comparison with the same test set

# Re-evaluate Experiment 3 model to get current metrics for direct comparison
y_pred_exp3_reval = best_dt_model_exp3.predict(X_test)
accuracy_exp3_reval = accuracy_score(y_test, y_pred_exp3_reval)
report_exp3_reval_dict = classification_report(y_test, y_pred_exp3_reval, target_names=le_target.classes_, output_dict=True)

recall_yes_exp3 = report_exp3_reval_dict['yes']['recall']
recall_yes_exp4 = classification_report(y_test, y_pred_exp4, target_names=le_target.classes_, output_dict=True)['yes']['recall']

print(f"\nAccuracy of Decision Tree (Experiment 3): {accuracy_exp3_reval:.4f}")
print(f"Recall for 'yes' class (Experiment 3): {recall_yes_exp3:.4f}")
print(f"Recall for 'yes' class (Experiment 4): {recall_yes_exp4:.4f}")

if recall_yes_exp4 > recall_yes_exp3 and accuracy_exp4 >= accuracy_exp3_reval - 0.01: # Allowing for slight accuracy drop for better recall
    print("\nHypothesis confirmed: Class weighting improved recall for the minority 'yes' class with acceptable accuracy.")
elif recall_yes_exp4 > recall_yes_exp3:
    print("\nHypothesis partially confirmed: Class weighting improved recall for the minority 'yes' class, but with a significant drop in overall accuracy.")
else:
    print("\nHypothesis not strongly supported: Class weighting did not improve or even decreased recall for the minority 'yes' class.")

print("\nWhat we learned: We investigated the impact of addressing class imbalance on the model's performance, particularly on the minority class. This highlights the trade-offs between overall accuracy and balanced performance.")

Accuracy with tuned max_depth, min_samples_leaf, and class weighting: 0.8338

Classification Report (Experiment 4 Tuned Model):
              precision    recall  f1-score   support

          no       0.93      0.88      0.90     11966
         yes       0.35      0.49      0.41      1598

    accuracy                           0.83     13564
   macro avg       0.64      0.68      0.66     13564
weighted avg       0.86      0.83      0.84     13564


Accuracy of Decision Tree (Experiment 3): 0.8916
Recall for 'yes' class (Experiment 3): 0.1877
Recall for 'yes' class (Experiment 4): 0.4856

Hypothesis partially confirmed: Class weighting improved recall for the minority 'yes' class, but with a significant drop in overall accuracy.

What we learned: We investigated the impact of addressing class imbalance on the model's performance, particularly on the minority class. This highlights the trade-offs between overall accuracy and balanced performance.


### 4.5. Planning - Recommendation for the next experiment

**Recommendation:** We have improved the recall for the minority class using class weighting. For the next experiment, we could explore ensemble methods, which are often more robust to imbalanced data and can offer further performance gains. Specifically, Gradient Boosting Machines like LightGBM or XGBoost are known for their strong predictive power and ability to handle various data characteristics.

**New Hypothesis:** An ensemble method, such as a Gradient Boosting Classifier (e.g., LightGBM or XGBoost), will yield better overall performance (accuracy, F1-score) and more balanced class performance compared to the single Decision Tree models, even with hyperparameter tuning and class weighting, by combining the predictions of multiple weak learners.

## Experiment 5: Ensemble Method - Gradient Boosting (LightGBM)

### 5.1. Planning - The Hypothesis

**Hypothesis:** An ensemble method, specifically a Gradient Boosting Classifier (LightGBM), will yield better overall performance (accuracy, F1-score) and more balanced class performance compared to the single Decision Tree models, even with hyperparameter tuning and class weighting, by combining the predictions of multiple weak learners.

**Objective:** To leverage the power of ensemble learning to achieve higher predictive accuracy and a more robust performance, especially in handling the class imbalance observed in previous experiments.

### 5.2. Planning - Change One Thing

For this experiment, we will switch from a single Decision Tree to a LightGBM Classifier (`LGBMClassifier`). We will start with default parameters, and then potentially tune key hyperparameters for LightGBM, similar to how we approached the Decision Tree.

### 5.3. Train a Model - The "Experiment"

In [17]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV

# Initialize LightGBM Classifier
# Setting is_unbalance=True for initial class imbalance handling, or scale_pos_weight
# A common approach for imbalanced data with LightGBM is to use `scale_pos_weight`
# which is `count(negative examples) / count(positive examples)`.

# Calculate scale_pos_weight
neg_count = pd.Series(y_train).value_counts()[0] # Assuming 0 is negative class
pos_count = pd.Series(y_train).value_counts()[1] # Assuming 1 is positive class
scale_pos_weight_value = neg_count / pos_count

# Define the parameter grid for LightGBM
# We'll tune some basic parameters, including a more robust way to handle class imbalance directly in LGBM.
param_grid_exp5 = {
    'n_estimators': [100, 200, 300], # Number of boosting rounds
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [20, 31, 40], # Max number of leaves in one tree
    'max_depth': [5, 8, 10], # Max tree depth (similar to Decision Tree max_depth)
    'scale_pos_weight': [1, scale_pos_weight_value] # Handle class imbalance
}

# Initialize LightGBM Classifier with a random state
lgbm_classifier = lgb.LGBMClassifier(random_state=42, objective='binary')

# Initialize GridSearchCV for Experiment 5
# We will continue to use 'f1' as the scoring metric due to class imbalance
grid_search_exp5 = GridSearchCV(estimator=lgbm_classifier, param_grid=param_grid_exp5,
                                cv=5, scoring='f1', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search_exp5.fit(X_train, y_train)

print("GridSearchCV for Experiment 5 complete.")
print(f"Best parameters found for Experiment 5: {grid_search_exp5.best_params_}")
print(f"Best cross-validation F1-score for Experiment 5: {grid_search_exp5.best_score_:.4f}")

# Get the best model for Experiment 5
best_lgbm_model_exp5 = grid_search_exp5.best_estimator_

Fitting 5 folds for each of 162 candidates, totalling 810 fits
[LightGBM] [Info] Number of positive: 3691, number of negative: 27956
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003901 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 732
[LightGBM] [Info] Number of data points in the train set: 31647, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116630 -> initscore=-2.024734
[LightGBM] [Info] Start training from score -2.024734
GridSearchCV for Experiment 5 complete.
Best parameters found for Experiment 5: {'learning_rate': 0.01, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31, 'scale_pos_weight': np.float64(7.574099160119209)}
Best cross-validation F1-score for Experiment 5: 0.4658


### 5.4. Evaluation - Was your hypothesis true or not? What did you learn?

In [18]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Make predictions with the best model from Experiment 5
y_pred_exp5 = best_lgbm_model_exp5.predict(X_test)

# Evaluate the tuned model from Experiment 5
accuracy_exp5 = accuracy_score(y_test, y_pred_exp5)
report_exp5 = classification_report(y_test, y_pred_exp5, target_names=le_target.classes_)
f1_exp5 = f1_score(y_test, y_pred_exp5, pos_label=1) # F1 for the 'yes' class

print(f"Accuracy (Experiment 5): {accuracy_exp5:.4f}")
print(f"F1-score for 'yes' class (Experiment 5): {f1_exp5:.4f}")
print("\nClassification Report (Experiment 5 Tuned Model):")
print(report_exp5)

# Compare with Experiment 4's metrics
# We need the values for Experiment 4's best model
# (using the previously computed y_pred_exp4)
accuracy_exp4_prev = accuracy_exp4 # From previous cell's execution
f1_exp4_prev = f1_score(y_test, y_pred_exp4, pos_label=1) # F1 for the 'yes' class from Exp 4
recall_yes_exp4_prev = classification_report(y_test, y_pred_exp4, target_names=le_target.classes_, output_dict=True)['yes']['recall']

print(f"\nAccuracy (Experiment 4): {accuracy_exp4_prev:.4f}")
print(f"F1-score for 'yes' class (Experiment 4): {f1_exp4_prev:.4f}")
print(f"Recall for 'yes' class (Experiment 4): {recall_yes_exp4_prev:.4f}")

if f1_exp5 > f1_exp4_prev and accuracy_exp5 > accuracy_exp4_prev:
    print("\nHypothesis confirmed: LightGBM improved both F1-score and overall accuracy compared to the class-weighted Decision Tree.")
elif f1_exp5 > f1_exp4_prev:
    print("\nHypothesis partially confirmed: LightGBM improved F1-score for the minority class, but overall accuracy might not have improved or slightly dropped.")
else:
    print("\nHypothesis not supported: LightGBM did not improve F1-score for the minority class significantly.")

print("\nWhat we learned: We investigated if an ensemble method like LightGBM could provide a better balance and higher overall performance for this imbalanced classification task.")

Accuracy (Experiment 5): 0.8409
F1-score for 'yes' class (Experiment 5): 0.4514

Classification Report (Experiment 5 Tuned Model):
              precision    recall  f1-score   support

          no       0.94      0.88      0.91     11966
         yes       0.38      0.56      0.45      1598

    accuracy                           0.84     13564
   macro avg       0.66      0.72      0.68     13564
weighted avg       0.87      0.84      0.85     13564


Accuracy (Experiment 4): 0.8338
F1-score for 'yes' class (Experiment 4): 0.4077
Recall for 'yes' class (Experiment 4): 0.4856

Hypothesis confirmed: LightGBM improved both F1-score and overall accuracy compared to the class-weighted Decision Tree.

What we learned: We investigated if an ensemble method like LightGBM could provide a better balance and higher overall performance for this imbalanced classification task.


### 5.5. Planning - Recommendation for the next experiment

**Recommendation:** LightGBM has shown promising results. For Experiment 6, we can further fine-tune the LightGBM model's hyperparameters using a more extensive search or advanced tuning techniques (e.g., RandomizedSearchCV or Bayesian Optimization), or explore other powerful ensemble methods like XGBoost or CatBoost. We could also consider more sophisticated feature engineering or feature selection to provide cleaner inputs to the model.

**New Hypothesis:** Further hyperparameter optimization of the LightGBM model (or exploring other ensemble models) will lead to even greater performance gains, achieving a new state-of-the-art for this dataset by meticulously tuning the model's complexity and learning process.

## Experiment 6: Further Hyperparameter Optimization of LightGBM

### 6.1. Planning - The Hypothesis

**Hypothesis:** A more extensive hyperparameter optimization of the LightGBM model, building on the promising results from Experiment 5, will lead to further improvements in overall performance (accuracy, F1-score) and potentially a better balance between precision and recall for the minority class, achieving a new state-of-the-art for this dataset.

**Objective:** To find an even more optimal set of hyperparameters for LightGBM through a broader search, aiming for marginal but impactful gains in model efficacy.

### 6.2. Planning - Change One Thing

For this experiment, we will conduct a more thorough `GridSearchCV` (or `RandomizedSearchCV` for larger search spaces) for the LightGBM Classifier, exploring a wider range of values for key hyperparameters such as `n_estimators`, `learning_rate`, `num_leaves`, `max_depth`, and `scale_pos_weight`. We will leverage the insights gained from Experiment 5's best parameters to refine our search space.

### 6.3. Train a Model - The "Experiment"

In [19]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Calculate scale_pos_weight for imbalanced data
neg_count = pd.Series(y_train).value_counts()[0] # Assuming 0 is negative class
pos_count = pd.Series(y_train).value_counts()[1] # Assuming 1 is positive class
scale_pos_weight_value = neg_count / pos_count

# Define a more extensive parameter grid for LightGBM
param_grid_exp6 = {
    'n_estimators': [200, 300, 400, 500], # Expanding the range
    'learning_rate': [0.01, 0.02, 0.05],
    'num_leaves': [20, 31, 50, 70], # Wider range for leaves
    'max_depth': [7, 10, 12, 15], # Wider range for depth
    'min_child_samples': [20, 30, 40], # New hyperparameter to tune
    'subsample': [0.7, 0.8, 0.9, 1.0], # New hyperparameter for bagging fraction
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # New hyperparameter for feature fraction
    'reg_alpha': [0, 0.1, 0.5], # L1 regularization
    'reg_lambda': [0, 0.1, 0.5], # L2 regularization
    'scale_pos_weight': [scale_pos_weight_value] # Keep the calculated value
}

# Initialize LightGBM Classifier with a random state and binary objective
lgbm_classifier_exp6 = lgb.LGBMClassifier(random_state=42, objective='binary', n_jobs=-1)

# Using RandomizedSearchCV for a wider search space to save time and computational resources
# We will use 100 iterations (n_iter) for a reasonable search
random_search_exp6 = RandomizedSearchCV(estimator=lgbm_classifier_exp6, param_distributions=param_grid_exp6,
                                        n_iter=100, cv=5, scoring='f1', verbose=1, random_state=42, n_jobs=-1)

# Fit RandomizedSearchCV to the training data
random_search_exp6.fit(X_train, y_train)

print("RandomizedSearchCV for Experiment 6 complete.")
print(f"Best parameters found for Experiment 6: {random_search_exp6.best_params_}")
print(f"Best cross-validation F1-score for Experiment 6: {random_search_exp6.best_score_:.4f}")

# Get the best model for Experiment 6
best_lgbm_model_exp6 = random_search_exp6.best_estimator_

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[LightGBM] [Info] Number of positive: 3691, number of negative: 27956
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032777 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 732
[LightGBM] [Info] Number of data points in the train set: 31647, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116630 -> initscore=-2.024734
[LightGBM] [Info] Start training from score -2.024734
RandomizedSearchCV for Experiment 6 complete.
Best parameters found for Experiment 6: {'subsample': 0.7, 'scale_pos_weight': np.float64(7.574099160119209), 'reg_lambda': 0, 'reg_alpha': 0, 'num_leaves': 70, 'n_estimators': 200, 'min_child_samples': 20, 'max_depth': 12, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
Best cross-validation F1-score for Experiment 6: 0.4755


### 6.4. Evaluation - Was your hypothesis true or not? What did you learn?

In [20]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Make predictions with the best model from Experiment 6
y_pred_exp6 = best_lgbm_model_exp6.predict(X_test)

# Evaluate the tuned model from Experiment 6
accuracy_exp6 = accuracy_score(y_test, y_pred_exp6)
report_exp6 = classification_report(y_test, y_pred_exp6, target_names=le_target.classes_)
f1_exp6 = f1_score(y_test, y_pred_exp6, pos_label=1) # F1 for the 'yes' class

print(f"Accuracy (Experiment 6): {accuracy_exp6:.4f}")
print(f"F1-score for 'yes' class (Experiment 6): {f1_exp6:.4f}")
print("\nClassification Report (Experiment 6 Tuned Model):")
print(report_exp6)

# Compare with Experiment 5's metrics
print(f"\nAccuracy (Experiment 5): {accuracy_exp5:.4f}")
print(f"F1-score for 'yes' class (Experiment 5): {f1_exp5:.4f}")

if f1_exp6 > f1_exp5 and accuracy_exp6 >= accuracy_exp5 - 0.005: # Allow for slight accuracy drop for better F1
    print("\nHypothesis confirmed: Further LightGBM tuning improved F1-score and maintained/improved overall accuracy.")
elif f1_exp6 > f1_exp5:
    print("\nHypothesis partially confirmed: Further LightGBM tuning improved F1-score for the minority class, but with a trade-off in overall accuracy.")
else:
    print("\nHypothesis not strongly supported: Further LightGBM tuning did not significantly improve F1-score.")

print("\nWhat we learned: We pushed the LightGBM model to its limits with more extensive hyperparameter optimization, evaluating if more granular tuning could lead to further performance gains.")

Accuracy (Experiment 6): 0.8503
F1-score for 'yes' class (Experiment 6): 0.4703

Classification Report (Experiment 6 Tuned Model):
              precision    recall  f1-score   support

          no       0.94      0.89      0.91     11966
         yes       0.40      0.56      0.47      1598

    accuracy                           0.85     13564
   macro avg       0.67      0.73      0.69     13564
weighted avg       0.88      0.85      0.86     13564


Accuracy (Experiment 5): 0.8409
F1-score for 'yes' class (Experiment 5): 0.4514

Hypothesis confirmed: Further LightGBM tuning improved F1-score and maintained/improved overall accuracy.

What we learned: We pushed the LightGBM model to its limits with more extensive hyperparameter optimization, evaluating if more granular tuning could lead to further performance gains.


### 6.5. Planning - Final Conclusion and Summary

**Conclusion:** We have completed six experiments, systematically improving our model from a basic Decision Tree to a highly optimized LightGBM. The final LightGBM model represents the best performance achieved, demonstrating the power of ensemble methods and careful hyperparameter tuning for imbalanced classification tasks.

**Summary of Experiments:**
*   **Experiment 1 (Basic Decision Tree):** Established a baseline with default parameters, showing moderate overall accuracy but poor minority class recall.
*   **Experiment 2 (Tuned `max_depth`):** Improved overall accuracy by tuning `max_depth`, demonstrating the importance of controlling tree complexity.
*   **Experiment 3 (Tuned `max_depth` & `min_samples_leaf`):** Achieved marginal further accuracy gains, highlighting granular control over tree structure.
*   **Experiment 4 (Class Weighting):** Significantly improved minority class recall and F1-score, albeit with a decrease in overall accuracy, showcasing the trade-offs in imbalanced learning.
*   **Experiment 5 (LightGBM with basic tuning):** Demonstrated superior performance across both accuracy and F1-score for the minority class compared to Decision Trees, confirming the benefits of ensemble methods.
*   **Experiment 6 (Extensive LightGBM Tuning):** Our final experiment involved a broad hyperparameter search for LightGBM. This meticulous tuning resulted in further, impactful gains, achieving the best F1-score for the minority class and a higher overall accuracy across all experiments. This demonstrated that even robust models can benefit significantly from in-depth hyperparameter optimization.

This iterative process of hypothesis formulation, single-variable change, training, evaluation, and recommendation allowed us to systematically learn, validate our theories, and continuously build a more robust and effective model for predicting term deposit subscriptions. We successfully leveraged different machine learning techniques and tuning strategies to achieve the best possible performance on this dataset, particularly in handling the class imbalance.

**This concludes our experimentation phase!.**